# Final RF300 Submission Workflow

Notebook wrapper for the final raw-feature RandomForest submission pipeline.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)


In [ ]:
random_state = 42
data_dir = "."
holdout_fraction = 0.2
output_dir = "artifacts_final_rf300"
submission_prefix = "final_rf300"
write_submission = False


In [ ]:
command = [
    sys.executable,
    str(ROOT / "scripts" / "train_final_rf300.py"),
    "--data-dir",
    data_dir,
    "--random-state",
    str(random_state),
    "--holdout-fraction",
    str(holdout_fraction),
    "--output-dir",
    output_dir,
    "--submission-prefix",
    submission_prefix,
]

if not write_submission:
    command.append("--skip-submission")

run = subprocess.run(
    command,
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)

raw_stdout = run.stdout
json_start = raw_stdout.index("{")
summary = json.loads(raw_stdout[json_start:])

print(raw_stdout)


In [ ]:
pd.DataFrame([
    {"feature_set": "raw", "feature_count": summary["feature_count"]},
])


In [ ]:
{
    "duplicate_target_groups": summary["duplicate_target_groups"],
    "constant_targets": summary["constant_targets"],
    "target_strategy": summary["target_strategy"],
}


In [ ]:
pd.DataFrame([
    {
        "train_rmse": summary["diagnostic_scores"]["train_rmse"],
        "validation_rmse": summary["diagnostic_scores"]["validation_rmse"],
        "full_train_rmse": summary["diagnostic_scores"]["full_train_rmse"],
    }
])


In [ ]:
feature_importance_df = pd.DataFrame([
    {"feature": feature, "importance": importance}
    for feature, importance in summary["feature_importance"].items()
]).sort_values("importance", ascending=False).reset_index(drop=True)
feature_importance_df


In [ ]:
summary
